# 🗓️ ConflictEnv GRPO Training V3.1 — "The Guidance Update"

**What this does:** Fixes the "Zero Gradient" problem by providing explicit command guidance and increasing sequence length.

### Changes in V3.1:
1. **System Prompt Injection:** Lists the 7 valid commands in every example.
2. **Increased Length:** `max_completion_length` 400 -> 600 to prevent JSON truncation.
3. **Reward Variance Fix:** Modified `reward_format_check` to ensure completions rarely tie.

---
## Cell 1: Install Dependencies
**IMPORTANT:** Run this once per session, then **Restart the Session**.

In [ ]:
# ============================================================
# CELL 1: INSTALL
# ============================================================
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl>=0.13.0" peft accelerate bitsandbytes
!pip install -q openenv-core gradio

import os, sys
!rm -rf /kaggle/working/MetaxBangalore
!git clone -q https://github.com/archittmittal/MetaxBangalore.git /kaggle/working/MetaxBangalore
sys.path.insert(0, "/kaggle/working/MetaxBangalore")

print("✅ Installation complete. RESTART session, then run Cell 2.")

---
## Cell 2: Load Model + LoRA

In [ ]:
# ============================================================
# CELL 2: MODEL LOADING
# ============================================================
import os, sys, re, json, requests
from datasets import Dataset
sys.path.insert(0, "/kaggle/working/MetaxBangalore")

hf_token = "hf_REMOVED"
os.environ["HF_TOKEN"] = hf_token

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
print("✅ Model loaded.")

---
## Cell 3: Dataset + Reward Functions

In [ ]:
# ============================================================
# CELL 3: DATASET & REWARDS
# ============================================================
from conflict_env.env import ConflictEnv
from conflict_env.models import ConflictAction, VALID_COMMANDS

SYSTEM_PROMPT = (
    "You are an Elite Executive Assistant. You MUST start with <thought> reasoning and end with a JSON action.\n"
    f"VALID COMMANDS: {', '.join(sorted(VALID_COMMANDS))}."
)

data = requests.get(
    "https://raw.githubusercontent.com/archittmittal/MetaxBangalore/main/training_prompts.json"
).json()

def format_prompt(example):
    return {
        "prompt": tokenizer.apply_chat_template(
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": example["prompt"]}
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
    }

train_dataset = Dataset.from_list(data).map(format_prompt).shuffle(seed=42)
print(f"✅ Dataset ready.")

def reward_format_check(prompts, completions, **kwargs):
    rewards = []
    for c in completions:
        r = 0.0
        if "<thought>" in c and "</thought>" in c: r += 0.2
        if "{" in c and "}" in c: r += 0.2
        try:
            m = re.search(r'\{[^{}]*"command"[^{}]*\}', c, re.DOTALL)
            if m:
                parsed = json.loads(m.group(0))
                if parsed.get("command", "").lower() in VALID_COMMANDS: r += 0.5
                else: r += 0.1
        except: pass
        r += min(0.05, 1.0 / (len(c) + 1))
        rewards.append(r)
    return rewards

def reward_conflict_resolution(prompts, completions, **kwargs):
    rewards = []
    for c in completions:
        try:
            m = re.search(r'\{[^{}]*"command"[^{}]*\}', c, re.DOTALL)
            if m:
                env = ConflictEnv()
                env.reset(task_name="auto")
                action = ConflictAction.model_validate_json(m.group(0))
                env.step(action)
                rewards.append(env.get_reward())
            else: rewards.append(0.01)
        except: rewards.append(0.01)
    return rewards

---
## Cell 4: Launch Training

In [ ]:
# ============================================================
# CELL 4: GRPO TRAINING
# ============================================================
from trl import GRPOTrainer, GRPOConfig

training_args = GRPOConfig(
    output_dir="/kaggle/working/conflict-env-grpo-v3-1",
    learning_rate=3e-5,
    num_train_epochs=1,
    max_steps=150,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=600,
    temperature=0.9,
    logging_steps=1,
    push_to_hub=True,
    hub_model_id="purvansh01/conflict-env-grpo-v3",
    hub_token=hf_token,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_format_check, reward_conflict_resolution],
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
)
trainer.train()

---
## Cell 7: Launch Interactive Gradio UI (FINAL MODEL)
Run this to get your final demo URL!

In [ ]:
# ============================================================
# CELL 7: GRADIO UI (FINAL)
# ============================================================
import os, sys, re, json
sys.path.insert(0, "/kaggle/working/MetaxBangalore")

import gradio as gr
from unsloth import FastLanguageModel
from conflict_env.models import VALID_COMMANDS

hub_model_name = "purvansh01/conflict-env-final"  # POINTING TO FINAL MODEL

print(f"🛰️ Launching Gradio for FINAL model: {hub_model_name}...")
ui_model, ui_tokenizer = FastLanguageModel.from_pretrained(
    model_name=hub_model_name,
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(ui_model)

def predict(scenario):
    sys_prompt = (
        "You are an Elite Executive Assistant. You MUST start with <thought> reasoning and end with a JSON action.\n"
        f"VALID COMMANDS: {', '.join(sorted(VALID_COMMANDS))}."
    )
    
    formatted_user_input = f"""Scenario: Custom Scenario
Details: {scenario}

Respond in the following format: <thought> reasoning </thought> {{"command": "...", "parameters": {{...}}}}"""
    
    prompt = ui_tokenizer.apply_chat_template(
        [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": formatted_user_input}
        ],
        tokenize=False,add_generation_prompt=True,
    )
    
    inputs = ui_tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = ui_model.generate(**inputs, max_new_tokens=500, temperature=0.7)
    response = ui_tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    
    thought = "Parsing failed. See Raw Output below."
    action = "Parsing failed. See Raw Output below."
    
    thought_match = re.search(r"<thought>(.*?)</thought>", response, re.DOTALL | re.IGNORECASE)
    if thought_match: thought = thought_match.group(1).strip()
    
    action_match = re.search(r"(\{.*?\})", response, re.DOTALL)
    if action_match: action = action_match.group(1).strip()
    
    return thought, action, response

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🗓️ ConflictEnv Agent Demo (FINAL)")
    gr.Markdown("This is the production-ready Conflict Resolution Assistant.")
    
    with gr.Row():
        with gr.Column():
            input_text = gr.Textbox(label="Describe the Conflict", placeholder="e.g. My flight is delayed...")
            submit_btn = gr.Button("Resolve Conflict", variant="primary")
        
    with gr.Row():
        thought_out = gr.Textbox(label="🧠 Agent Reasoning", lines=8)
        action_out = gr.Code(label="🛠️ Planned Action (JSON)", language="json")
    
    raw_out = gr.Textbox(label="📝 Raw Model Output (Debug)", lines=4)

    submit_btn.click(predict, inputs=input_text, outputs=[thought_out, action_out, raw_out])
    
demo.launch(share=True)